In [1]:
import pandas as pd
import torch
import torch.nn.functional as F # type: ignore
from transformers import AutoTokenizer, AutoModelForSequenceClassification # type: ignore


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



Usando il dispositivo: cuda


In [ ]:

tokenizer = AutoTokenizer.from_pretrained("rafalposwiata/deproberta-large-depression")
model = AutoModelForSequenceClassification.from_pretrained("rafalposwiata/deproberta-large-depression",).to(device)




tokenizer_config.json: 0.00B [00:00, ?B/s]

c:\Users\Franco\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Franco\.cache\huggingface\hub\models--rafalposwiata--deproberta-large-depression. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/924 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

In [ ]:

def classify_text(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
    probs = F.softmax(logits, dim=-1).squeeze().tolist()
    label = torch.argmax(logits, dim=-1).item()
    return label, probs[0], probs[1], probs[2]

In [ ]:

df = pd.read_csv(r"C:\Users\Franco\Desktop\Pubblicazione\16\eRisk.csv")

In [ ]:

df[['Predicted_Label', 'Prob_Not_Depressed', 'Prob_Moderate_Depressed', 'Prob_Severe_Depressed']] = df['Text'].apply(lambda x: pd.Series(classify_text(x)))


In [ ]:

df.rename(columns={'Classification': 'Label_User'}, inplace=True)

In [ ]:

df.to_csv(r"C:\Users\Franco\Desktop\Pubblicazione\16\Kaggle_classification\output_Classification.csv", index=False)